# Kaj se je zgodilo z IT in odpuščanji po koronavirusu?
### Zgodba v grafih

To je zgodba o tem, kaj se dogaja v tehnološkem svetu od leta 2020: **kdaj** so podjetja napovedovala odpuščanja, **zakaj** morda ravno takrat, **katera podjetja** so bila najbolj na udaru, **kaj iščejo delodajalci**, **kje** v ZDA se je delo spremenilo in **kako se je odzvala borza**.

Vsi grafi so interaktivni: miško premakni na točko ali črto in vidiš točne vrednosti; z vlečenjem približaš del grafa, z dvoklikom se vrneš nazaj.

**Kako brati to zgodbo.** Pod vsakim grafom so štirje kratki odgovori:

- **Kaj vidimo?** Kako se graf bere.
- **Zakaj smo to naredili?** Katero vprašanje nas je zanimalo.
- **Kaj to pomeni?** Kaj kažejo podatki, s številkami, izračunanimi iz podatkov.
- **Česa ne vemo?** Kaj podatki ne morejo pokazati. To je pomemben del: večinoma vidimo, kaj se je dogajalo *ob istem času*, ne pa *zakaj*.

**Odkod podatki.** Seznam napovedanih odpuščanj (Layoffs.fyi), javni računovodski podatki podjetij (ameriška komisija SEC), obrestne mere in borzni indeks (FRED), uradna statistika trga dela (BLS), mesečne objave "Who is hiring?" na Hacker Newsu in cene delnic (Yahoo Finance).

In [ ]:
from pathlib import Path
import os, sys
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "build_master.py").exists())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)          # poti do podatkov so relativne glede na mapo projekta
import plotly.io as pio
from analysis import viz, story_text as tx
from analysis.viz_theme import explanation_markdown

# Grafi se vidijo v JupyterLab in v izvoženem HTML (ta potrebuje internet, da naloži plotly.js).
# Za izvoz, ki dela brez interneta, zamenjaj z "notebook" (datoteka je večja za okoli 4 MB).
pio.renderers.default = "plotly_mimetype+notebook_connected"

story = viz.prepare_story_data()                       # zažene obstoječe analize (nekaj minut)
figs = viz.build_figures(story)
explain = tx.build_explanations(story)

try:                                                   # poglavji 4 in 5 (Hacker News in zemljevid) pripravi drugi del ekipe
    from analysis import viz_text_geo as vtg
    for name, (fig, parts) in vtg.build_chapter_4_5().items():
        figs[name], explain[name] = fig, parts
except Exception:
    pass


def show(name):
    """Graf in pod njim štiri razlage v preprostem jeziku."""
    figs[name].show()
    display(Markdown(explanation_markdown(explain[name])))


def show_chapter(order, prefix, missing_text):
    names = [n for n in order if n in figs] + sorted(k for k in figs if k.startswith(prefix) and k not in order)
    if not names:
        display(Markdown(missing_text))
    for n in names:
        show(n)

## 1. Kdaj so podjetja napovedovala odpuščanja?

Začnimo s preprostim vprašanjem: kdaj? Prvi graf postavi na isto os več stvari, ki jih običajno gledamo ločeno. Drugi pokaže, kako se napovedi razporejajo po tednih v letu.

In [ ]:
show("1_casovnica")

In [ ]:
show("1_koledar")

## 2. Zakaj ravno takrat?

Zdaj vprašanje "zakaj". Preverimo tri pogoste razlage: dražji denar in padec borze, dva znana datuma (prvo zvišanje obrestne mere in izid ChatGPT) ter prezaposlovanje v času razcveta. Pri vsaki povemo, kaj smo preizkusili in česa podatki ne morejo pokazati.

In [ ]:
show("2_zamiki")

In [ ]:
show("2_datumi")

In [ ]:
show("2_rast")

## 3. Katera podjetja se znajdejo na seznamu?

Naslednje vprašanje: ali lahko po javnih računovodskih podatkih podjetja vnaprej ocenimo, katera bodo napovedala odpuščanja, in kaj to ocenjevanje poganja?

In [ ]:
show("3_velikost")

In [ ]:
show("3_podjetja")

In [ ]:
show("3_modeli")

In [ ]:
show("3_skupine")

## 4. Kaj iščejo delodajalci?

Pogled z druge strani: ne kdo odpušča, ampak kaj delodajalci iščejo. Uporabili smo mesečne objave "Who is hiring?" na Hacker Newsu, kjer podjetja objavljajo prosta delovna mesta. To je zanimiv, a poseben vzorec: gre predvsem za startupe in tehnološka podjetja, zato ne govori za vse delodajalce.

In [ ]:
show_chapter(["4_teme_povrsina", "4_teme_mali_grafi", "4_teme_toplotna", "4_daljava_ui"], "4_",
             "*Poglavje o objavah delodajalcev bomo dodali, ko bodo podatki priključeni.*")

## 5. Kje v ZDA?

Še zemljevid: kako se je med letoma 2019 in 2024 spremenilo število delovnih mest v IT po ameriških metropolitanskih območjih. Pozor: IT je tu ozko opredeljen (računalniško programiranje, svetovanje in sorodne storitve), podatki za nekatera območja pa so zaradi zaupnosti skriti. Skrit podatek ni enak nič.

In [ ]:
show_chapter(["5_zemljevid", "5_regije"], "5_", "*Zemljevid bomo dodali, ko bodo podatki priključeni.*")

## 6. Kaj naredi borza?

Za konec še borza: ali vlagatelji objavo odpuščanj nagradijo ali kaznujejo? Pogledali smo gibanje cene delnice okoli objav, ko odštejemo gibanje celotne borze.

In [ ]:
show_chapter(["6_pot", "6_porazdelitev", "6_skupine"], "6_",
             "*Ta del bo na voljo, ko bodo prenesene cene delnic (python collect.py --sources prices).*")

## Povzetek

In [ ]:
povzetek = tx.build_summary(story)
display(Markdown("### Kaj smo se naučili\n\n" + "\n\n".join(f"- {x}" for x in povzetek["zakljucki"])))
display(Markdown("### Česa ne vemo\n\n" + "\n\n".join(f"- {x}" for x in povzetek["odprto"])))

## Viri

- Layoffs.fyi: seznam napovedanih odpuščanj v tehnološkem sektorju.
- US Securities and Exchange Commission (EDGAR): javni računovodski podatki podjetij.
- Federal Reserve Bank of St. Louis (FRED): obrestna mera Fed in indeks NASDAQ Composite.
- US Bureau of Labor Statistics: zaposlenost po panogah (QCEW), statistika odpuščanj (JOLTS).
- Hacker News: mesečne objave "Ask HN: Who is hiring?".
- US Census Bureau: podatki o metropolitanskih območjih.
- Yahoo Finance (prek paketa yfinance): dnevne cene delnic, samo za raziskovalno rabo.